# Entropy - OLMo 2 Model Family

### Olmo 2 - 1B
model_name = OLMo-2-0425-1B  
revision = stage1-step990000-tokens2077B

### Olmo 2 - 7B
model_name = OLMo-2-1124-7B  
revision = stage1-step99000-tokens416B

### Olmo 2 - 13B
model_name = OLMo-2-1124-13B  
revision = stage1-step99000-tokens831B 

## Setup

In [16]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Local


In [17]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [18]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [ ]:
from pathlib import Path
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent
    print(PROJECT_ROOT)

In [8]:
# Cell 3: Imports
import torch
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [9]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import src
from src.tl217_olmo2_adapter import load_olmo2_tl217

In [10]:
model_name = "OLMo-2-0425-1B" 
revision = "stage1-step990000-tokens2077B"
short_name = model_name.split('/')[-1] 

In [11]:
# Cell 6: Load Model
model = load_olmo2_tl217("allenai/OLMo-2-0425-1B", device=device, revision=revision)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

/opt/homebrew/Caskroom/miniconda/base/envs/mechinterp/lib/python3.11/site-packages/transformer_lens/HookedTransformerConfig.py:343: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.10.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  utils.warn_if_mps(self.device)
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OLMo 2 TL2.17 validation
  max |logit difference|:  5.29289e-05
  mean |logit difference|: 5.46584e-06
  top-1 match at every position: True
Loaded pretrained model allenai/OLMo-2-0425-1B into TransformerLens 2.17.0
Layers: 16
Heads: 16
Hidden size: 2048
Params: 1485.3M


### Entropy

In [12]:
from pathlib import Path
import torch
import yaml
import pandas as pd
# Cell: Compute entropy (keep as a function — it's pure math)
def compute_entropy(logits):
    """H = -Σ P(x_i) log P(x_i)"""
    probs = torch.nn.functional.softmax(logits[0], dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=-1)
    return entropy

In [13]:
# Cell: Run entropy analysis
prompts_path = PROJECT_ROOT / 'data' / 'all_prompts.yml'
with open(prompts_path, 'r') as f:
    templates = yaml.safe_load(f)

results = []

for case in templates['prompts']:
    prompt = case['prompt']
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    entropy = compute_entropy(logits)

    results.append({
        'prompt_id': case['prompt_id'],
        'concept': case['concept'],
        'prompt_type': case['prompt_type'],
        'template_type': case['template_type'],
        'prompt': prompt,
        'n_tokens': len(entropy),
        'mean_entropy': round(entropy.mean().item(), 4),
        'last_token_entropy': round(entropy[-1].item(), 4),
        'max_entropy': round(entropy.max().item(), 4),
        'min_entropy': round(entropy.min().item(), 4),
        'model': model_name
    })

entropy_df = pd.DataFrame(results)
entropy_df

,prompt_id,concept,prompt_type,template_type,prompt,n_tokens,mean_entropy,last_token_entropy,max_entropy,min_entropy,model
0,decl_screen_reader_001,screen reader,declarative,cloze,A screen reader is,5,5.2419,2.1065,8.1890,2.1065,OLMo-2-0425-1B
1,decl_wcag_001,WCAG,declarative,cloze,WCAG stands for,5,3.5697,1.3924,7.1760,0.1445,OLMo-2-0425-1B
2,decl_skip_link_001,skip link,declarative,cloze,A skip link is,5,5.5844,2.1489,8.1890,2.1489,OLMo-2-0425-1B
3,decl_alt_text_001,alt text,declarative,cloze,The purpose of alt text is,7,3.8682,0.7946,8.5978,0.2614,OLMo-2-0425-1B
4,decl_aria_001,ARIA,declarative,cloze,ARIA stands for,5,4.9705,5.8377,7.1760,0.2690,OLMo-2-0425-1B
...,...,...,...,...,...,...,...,...,...,...,...
87,ctrl_bicycle_cloze_001,bicycle,control,cloze,A bicycle is used for,6,4.8879,3.5527,8.1890,2.0981,OLMo-2-0425-1B
88,ctrl_bicycle_direct_001,bicycle,control,direct_question,What is a bicycle?,6,5.8958,4.4229,8.5783,4.2446,OLMo-2-0425-1B
89,ctrl_bicycle_instruction_001,bicycle,control,instruction,Explain bicycles to a web developer.,9,4.8719,5.4560,7.1760,3.2641,OLMo-2-0425-1B
90,ctrl_bicycle_evaluative_001,bicycle,control,evaluative,A bicycle without brakes is not safe because,9,4.6029,3.0498,8.1890,3.0436,OLMo-2-0425-1B


In [14]:
# Cell: Save results

output_dir = PROJECT_ROOT / 'results' / 'entropy' / 'olmo' / model_name
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{short_name}-entropy.csv'
entropy_df.to_csv(output_path, index=False)
print(f"Saved {len(entropy_df)} entropy measurements to {output_path}")

Saved 92 entropy measurements to /Users/trishasalas/Repos/Research/tmlr/results/entropy/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-entropy.csv


In [15]:
output_dir = PROJECT_ROOT / 'results' / 'entropy' / 'olmo' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / f'{model_name}-entropy.md', 'w') as f:
  f.write(f"# Model data captured during Entropy Battery\n")
  f.write(f"- Model name: {model_name}\n")
  f.write(f"- Model dtype: {next(model.parameters()).dtype}\n")
  f.write(f"- Layers: {model.cfg.n_layers}\n")
  f.write(f"- Heads: {model.cfg.n_heads}\n")
  f.write(f"- Hidden size: {model.cfg.d_model}\n")
  f.write(f"- Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")



print(f"Saved to {output_dir}/{model_name}-model-dtype.md")

Saved to /Users/trishasalas/Repos/Research/tmlr/results/entropy/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-model-dtype.md


In [41]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "entropy results: {model_name}"
!git push

[main ced74f4] entropy results: gpt2-large
 12 files changed, 300 insertions(+)
 create mode 100644 results/entropy/gpt2/gpt2-large/gpt2-large-elicitation-entropy-binding.md
 create mode 100644 results/entropy/gpt2/gpt2-large/gpt2-large-entropy.csv
 create mode 100644 results/entropy/gpt2/gpt2-medium/gpt2-medium-elicitation-entropy-binding.md
 create mode 100644 results/entropy/gpt2/gpt2-medium/gpt2-medium-entropy.csv
 create mode 100644 results/entropy/gpt2/gpt2/gpt2-elicitation-entropy-binding.md
 create mode 100644 results/entropy/gpt2/gpt2/gpt2-entropy.csv
 rename results/entropy/pythia/{ => pythia-12b}/pythia-12b-entropy.csv (100%)
 rename results/entropy/pythia/{ => pythia-160m}/pythia-160m-entropy.csv (100%)
 rename results/entropy/pythia/{ => pythia-1b}/pythia-1b-entropy.csv (100%)
 rename results/entropy/pythia/{ => pythia-2.8b}/pythia-2.8b-entropy.csv (100%)
 rename results/entropy/pythia/{ => pythia-410m}/pythia-410m-entropy.csv (100%)
 rename results/entropy/pythia/{ => pyt

### Delete Model & Clear Cache

In [42]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared
